# GraphRAG-Bench — pilote de récupération sémantique sur Colab

**Objectif :** reproduire l'audit (4 072 questions), valider les tests, retrouver des passages avec BGE-small et BGE-large, puis comparer **TF-IDF et BGE sur les mêmes chunks** avec une feuille de vérification des preuves.

**Pré-requis :** dans Colab, **Exécution → Modifier le type d'exécution → T4 GPU**. Exécutez les cellules dans l'ordre. Le code utilise la branche `feature/graphrag-pilot-rag-retrieval` du fork `MamadyIA/GraphRAG-Benchmark`.

Ce notebook effectue uniquement de la **récupération de passages** : il ne génère aucune réponse et ne mesure aucune métrique de benchmark. Les 116 Go de stockage et la mémoire GPU sont temporaires et les quotas gratuits ne sont pas garantis.

## 1 — Vérifier les ressources de la session

In [ ]:
import torch, psutil, shutil
print("RAM totale (Go):", round(psutil.virtual_memory().total / 1024**3, 2))
print("Disque libre (Go):", round(shutil.disk_usage("/content").free / 1024**3, 2))
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM (Go):", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("Activez un runtime GPU dans les paramètres Colab avant de continuer.")

## 2 — Récupérer les scripts sur GitHub et installer Sentence Transformers

La branche contient l'audit, le prototype de récupération et leurs tests. Il n'est pas nécessaire de téléverser le fichier pilote local : nous le recréons de façon déterministe avec la graine 42.

In [ ]:
from pathlib import Path
import subprocess, sys

BRANCH = "feature/graphrag-pilot-rag-retrieval"
REPO = Path("/content/GraphRAG-Benchmark")
if not REPO.exists():
    subprocess.run([
        "git", "clone", "--depth", "1", "--branch", BRANCH,
        "https://github.com/MamadyIA/GraphRAG-Benchmark.git", str(REPO)
    ], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
import sentence_transformers
print("Dépôt:", REPO)
print("Sentence Transformers:", sentence_transformers.__version__)
print("CUDA toujours disponible:", torch.cuda.is_available())
assert torch.cuda.is_available(), "Le GPU T4 n'est plus disponible."

## 3 — Reproduire l'audit et exécuter les tests

Les sorties sont placées dans `/content/graphrag_audit_results`, **hors du dépôt**. Aucun dataset n'est modifié.

In [ ]:
import json

commands = [
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_benchmark_data_audit.py", "-v"],
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_pilot_retrieval.py", "-v"],
    [sys.executable, "tools/benchmark_data_audit.py", "--root", ".", "--seed", "42"],
]
for command in commands:
    subprocess.run(command, cwd=REPO, check=True)

RESULTS = REPO.parent / "graphrag_audit_results"
pilot = json.loads((RESULTS / "pilot_12_questions.json").read_text(encoding="utf-8"))
report = json.loads((RESULTS / "data_quality_report.json").read_text(encoding="utf-8"))
assert report["rows_checked"] == 4072
assert report["excluded_rows"] == 4
assert len(pilot) == 12
assert any(item["id"] == "Novel-3d21bfb3" for item in pilot)
print("VALIDATION: 4072 questions, 4 exclusions, 12 questions pilotes")
print("Pilote:", [p["id"] for p in pilot])

## 4 — Premier test GPU, BGE-small sur une question littéraire

Nous commençons par un seul roman pour contrôler le chargement des modèles et l'indexation. Ni les réponses de référence ni les preuves attendues ne sont utilisées pour produire les embeddings.

In [ ]:
import sys
sys.path.insert(0, str(REPO / "experiments"))
from pilot_retrieval import retrieve

one = [item for item in pilot if item["id"] == "Novel-3d21bfb3"]
assert len(one) == 1
first_result = retrieve(REPO, one, mode="semantic", model_name="BAAI/bge-small-en-v1.5", chunk_size=256, overlap=32, top_k=5)
(RESULTS / "pilot_one_bge_small.json").write_text(json.dumps(first_result, ensure_ascii=False, indent=2), encoding="utf-8")
print("ID:", first_result[0]["id"])
print("Question:", first_result[0]["question"])
for chunk in first_result[0]["retrieved_chunks"]:
    print(f"Rang {chunk['rank']}, score {chunk['score']}: {chunk['text'][:160]}...")
assert len(first_result[0]["retrieved_chunks"]) == 5
assert first_result[0]["generated_answer"] is None

## 5 — Pilote de 12 questions avec BGE-large

**N'exécutez cette cellule qu'après la réussite de la cellule 4.** Elle télécharge BGE-large dans le runtime Colab, indexe les cinq sources sollicitées par le pilote et sauvegarde les cinq passages par question. Le GPU T4 attribué devrait pouvoir exécuter cet essai ; la disponibilité réelle dépend néanmoins de l'état du runtime.

In [ ]:
output = RESULTS / "pilot_bge_large_retrieval.json"
subprocess.run([
    sys.executable, "experiments/pilot_retrieval.py",
    "--root", ".",
    "--pilot", str(RESULTS / "pilot_12_questions.json"),
    "--output", str(output),
    "--mode", "semantic",
    "--model", "BAAI/bge-large-en-v1.5",
    "--chunk-size", "256", "--overlap", "32", "--top-k", "5",
], cwd=REPO, check=True)
retrieved = json.loads(output.read_text(encoding="utf-8"))
assert len(retrieved) == 12
assert all(len(item["retrieved_chunks"]) == 5 for item in retrieved)
print("BGE-LARGE: 12 questions et 60 passages récupérés")
for item in retrieved:
    print(item["id"], item["domain"], item["question_type"], "Top1", item["retrieved_chunks"][0]["score"])

## 6 — Comparaison équitable : mêmes passages, même découpage, même Top-5

Nous recalculons le TF-IDF et BGE-large sur **exactement les mêmes segments de 256 tokens**, avec 32 tokens de chevauchement. Aucun score TF-IDF n'est comparé numériquement à un score BGE. Le fichier CSV produit une grille pour vérifier manuellement les preuves ; **aucune métrique de rappel ou d'exactitude n'est calculée automatiquement**.

Cette cellule récupère les nouveaux scripts sur la branche, exécute les nouveaux tests et lance la comparaison sur le GPU T4.

In [ ]:
from pathlib import Path
import subprocess, sys, json, torch

REPO = Path("/content/GraphRAG-Benchmark")
RESULTS = REPO.parent / "graphrag_audit_results"
assert torch.cuda.is_available(), "Activer un runtime Colab T4 avant ce test."
assert (RESULTS / "pilot_12_questions.json").exists(), "Exécuter d'abord les cellules 1 à 3."

subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", "test_pilot_fair_compare.py", "-v"], cwd=REPO, check=True)

subprocess.run([
    sys.executable, "experiments/pilot_fair_compare.py",
    "--root", ".", "--pilot", str(RESULTS / "pilot_12_questions.json"),
    "--output-dir", str(RESULTS), "--model", "BAAI/bge-large-en-v1.5",
    "--chunk-size", "256", "--overlap", "32", "--top-k", "5",
    "--device", "cuda",
], cwd=REPO, check=True)

report = json.loads((RESULTS / "pilot_shared_comparison.json").read_text(encoding="utf-8"))
print("Questions comparées:", len(report["comparison"]))
print("Sources indexées:", report["sources_indexed"])
print("Segments communs au top-5 (total sur 12 questions):", sum(x["shared_top_k_chunks"] for x in report["comparison"]))
for item in report["comparison"]:
    print(item["id"], "TF-IDF Top1:", item["lexical_top1_chunk"], "BGE Top1:", item["semantic_top1_chunk"], "passages communs:", item["shared_top_k_chunks"])
print("Revue manuelle:", RESULTS / "pilot_shared_manual_review.csv")

## 7 — Sauvegarder les résultats dans une archive ZIP

Le CSV contient 120 passages (12 questions × 5 passages × 2 méthodes), sans note attribuée automatiquement. Vous pourrez le relire dans Excel et renseigner la colonne `supports_evidence_yes_no_partial` avec `yes`, `no` ou `partial`.

La sortie ZIP est copiée sur votre ordinateur avant la fermeture de la session Colab.

In [ ]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

RESULTS = Path("/content/graphrag_audit_results")
archive = RESULTS / "GraphRAG_Pilot_Comparaison.zip"
filenames = [
    "data_quality_report.json", "pilot_12_questions.json",
    "pilot_one_bge_small.json", "pilot_bge_large_retrieval.json",
    "pilot_tfidf_shared_retrieval.json", "pilot_bge_shared_retrieval.json",
    "pilot_shared_comparison.json", "pilot_shared_manual_review.csv",
]
with ZipFile(archive, "w", compression=ZIP_DEFLATED) as z:
    for filename in filenames:
        path = RESULTS / filename
        if path.exists():
            z.write(path, arcname=filename)
        else:
            print("Absent:", filename)
print("Archive enregistrée:", archive, "taille (Mo):", round(archive.stat().st_size / 1024**2, 2))
files.download(str(archive))

## Interprétation scientifique

- La validation du pilote est **structurelle**, non une vérification humaine exhaustive des vérités terrain.
- BGE-small et BGE-large produisent des passages, pas des réponses générées. Le prochain lot devra intégrer un LLM et les métriques de récupération.
- Les anciens résultats TF-IDF découpaient en **256 mots**, tandis que BGE utilisait **256 tokens** : ces anciens rangs/scores ne permettent pas une comparaison quantitative rigoureuse.
- **La nouvelle comparaison** recalcule les deux méthodes avec **le même tokenizer et les mêmes passages**, mais une validation des preuves est indispensable pour mesurer la pertinence.
- Le nombre de passages communs au top-5 mesure un **accord entre méthodes**, pas l'Evidence Recall.
- Ne pas comparer numériquement les scores TF-IDF et BGE : les échelles diffèrent.
- Le stockage Colab est temporaire : télécharger le ZIP avant la fin de session.